<a href="https://colab.research.google.com/github/sohnaamie/gambia-semantic-segmentation/blob/amie-deeplab/notebooks/deeplab_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

In [ ]:
device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

print(device)

In [ ]:
model = torchvision.models.segmentation.deeplabv3_resnet50(
    weights='DEFAULT'
)

model.to(device)

model.eval()

In [ ]:
transform = transforms.Compose([

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [ ]:
import requests
from PIL import Image
from io import BytesIO

url = "https://images.unsplash.com/photo-1449824913935-59a10b8d2000"

response = requests.get(url)

image = Image.open(
    BytesIO(response.content)
).convert('RGB')

plt.figure(figsize=(10,6))
plt.imshow(image)
plt.axis('off')
plt.show()

In [ ]:
input_tensor = transform(image)

input_batch = input_tensor.unsqueeze(0).to(device)

with torch.no_grad():

    output = model(input_batch)['out'][0]

prediction = output.argmax(0)

print(np.unique(
    prediction.cpu().numpy()
))

In [ ]:
plt.figure(figsize=(10,8))

plt.imshow(
    prediction.cpu().numpy(),
    cmap='tab20'
)

plt.colorbar()

plt.title(
    "DeepLabV3+ Baseline Prediction"
)

plt.axis('off')

plt.show()

In [ ]:
pred_np = prediction.cpu().numpy()

print(np.unique(pred_np))

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(14,6))

ax[0].imshow(image)
ax[0].set_title("Original Image")
ax[0].axis('off')

ax[1].imshow(prediction.cpu().numpy(), cmap='tab20')
ax[1].set_title("DeepLab Prediction")
ax[1].axis('off')

plt.show()

In [ ]:
scores = torch.softmax(output, dim=0)

print(scores.max())
print(scores.min())